# Preparation

## Import modules

In [1]:
# Cell 1: Mount & Navigate
from google.colab import drive
from torch.version import cuda

drive.mount('/content/drive')

# Go to correct folder
%cd /content/drive/MyDrive/Colab\ Notebooks/thesis/LSTM_Train

# Verify structure
!ls -la ../
# Should show: dataset/  thesis_utils/  LSTM_Train/

!pip install loguru torchxlstm fastparquet

import sys
from pathlib import Path

# Add thesis_utils to path (parent dir)
sys.path.insert(0, '/content/drive/MyDrive/Colab\ Notebooks/thesis')

# Or simpler:
sys.path.insert(0, str(Path.cwd().parent))

Added to path: /Users/gabriel/Documents/01_TUW/00_Thesis/trade_under_pressure_thesis/src
thesis_utils exists: True


In [ ]:
# Prediction using LSTM, GRU-LSTM, xLSTM
import copy
import math
import os
from typing import List

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils import clip_grad_norm_
from torch.optim import Optimizer
from torch.optim.lr_scheduler import LRScheduler
from torch.utils.data import DataLoader, Dataset, Subset

from sklearn.model_selection import KFold, GroupShuffleSplit

import thesis_utils as tu

from pandas import DataFrame

from torch.amp import GradScaler, autocast

In [ ]:
CHKPT_DIR = "/content/checkpoints"
os.makedirs(CHKPT_DIR, exist_ok=True)

In [ ]:
LOSS_DIR = "/content/drive/MyDrive/Colab Notebooks/thesis/GRU_Train/"
LOSS_LOG_PATH = os.path.join(LOSS_DIR, "train_loss_GRU.csv")

with open(LOSS_LOG_PATH, "w") as f:
  f.write("epoch,train_loss,val_loss\n")

In [ ]:
def ckpt_path(serial, fold):
  return os.path.join(CHKPT_DIR, f"{serial}_fold{fold}.pt")

In [ ]:
def safe_load_ckpt(path: str, map_location="cpu"):
  """Load a torch checkpoint, but delete it and return None if corrupted."""
  if not os.path.exists(path):
    return None

  try:
    return torch.load(path, map_location=map_location)
  except Exception as e:
    print(f"⚠️ Corrupted checkpoint detected: {path}")
    print(f"   Error: {e}")
    # Delete immediately so it cannot be loaded again later
    try:
      os.remove(path)
      print("   Deleted corrupted checkpoint.")
    except Exception as del_e:
      print(f"   Could not delete checkpoint: {del_e}")
    return None

## Configuration

In [2]:
# Model parameters
HORIZON = 1
BATCH_SIZE = 2048
NUM_EPOCHS = 100
HIDDEN_SIZE = 512
N_LAYERS = 3
DROPOUT = 0.05
EMBEDDING_SIZE = 256

# Train parameters
TARGET = "EXPORT_centered"
FEATURES = [
  "contig", "comlang_off", "colony", "smctry",
]
N_SPLITS = 8
PATIENCE = 10
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-3
RANDOM_SEED = 16
N_LAGS = 5
SUBSAMPLE_ENABLED = True
N_DYADS = 10

SANCTION_COLS = ["arms", "military", "trade", "travel", "other", "financial"]

# Torch config
torch.manual_seed(RANDOM_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------
# Colab / L4 performance knobs
# ----------------------------
if device.type == "cuda":
  print("GPU:", torch.cuda.get_device_name(0))
  print("CUDA:", torch.version.cuda, "| PyTorch:", torch.__version__)
  !nvidia-smi -L

  # cuDNN autotuner (best when shapes are stable, typical in training)
  torch.backends.cudnn.benchmark = True

  # Better GEMM kernels on Ampere+ (L4 is Ada)
  torch.set_float32_matmul_precision("high")

  # Reduce allocator fragmentation for long runs
  os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Mixed precision (L4 supports bf16 well)
USE_AMP = True
AMP_DTYPE = torch.bfloat16 if (
    device.type == "cuda" and getattr(torch.cuda, "is_bf16_supported", lambda: False)()) else torch.float16

# torch.compile can improve throughput (PyTorch 2.0+)
USE_TORCH_COMPILE = True
COMPILE_MODE = "max-autotune"  # alternatives: "reduce-overhead", "default"

# GradScaler is needed for fp16; for bf16 it should be disabled
USE_GRAD_SCALER = (device.type == "cuda" and USE_AMP and AMP_DTYPE == torch.float16)

print("Using device:", device)
if torch.cuda.is_available():
  print(torch.cuda.get_device_name(0))
  # Enable TF32 for faster computing on Ampere+ GPUs
  torch.backends.cuda.matmul.allow_tf32 = True
  torch.backends.cudnn.allow_tf32 = True

dyads_case_study = [
  # "USA_CHN", "CHN_USA",
  # "USA_CAN", "CAN_USA",
  # "DEU_CHN", "CHN_DEU",
  # "USA_DEU", "DEU_USA",
  # "USA_MEX", "MEX_USA",
  # "AUS_CHN", "CHN_AUS",
  # "USA_JPN", "JPN_USA",
  # "DEU_JPN", "JPN_DEU",
  # "USA_AUS", "AUS_USA",
  # "DEU_RUS", "RUS_DEU",
]

Device type:  mps


In [3]:
# Save config
SAVE_ENABLED = True
SERIAL_NUMBER = f"GRU-{LEARNING_RATE}lr-{DROPOUT}d-{HIDDEN_SIZE}hs-{WEIGHT_DECAY}wd-{BATCH_SIZE}bs-{N_LAYERS}layers-{EMBEDDING_SIZE}es-kfolds{N_SPLITS}-hp"
SERIAL_NUMBER = SERIAL_NUMBER.replace(".", "_")
PATH_TO_FOLDER = ""

GRU-0_0003lr-0_05d-256hs-0wd-128bs-2layers-128es-kfolds8-hp


## Load Data

In [4]:
processed = pd.read_parquet(path="../dataset/processed.parquet", engine="fastparquet")
df: DataFrame = processed.copy(deep=True)

### Sort, shift and compute data

In [5]:
# Create dyad_id column
df["dyad_id"] = df["ISO3_reporter"] + "_" + df["ISO3_partner"]
df = df.sort_values(by=["dyad_id", "Year"], ignore_index=True)

In [6]:
# Remove case study dyad_pairs
mask_keep = ~np.isin(df["dyad_id"], dyads_case_study)
df = df.loc[mask_keep].reset_index(drop=True)

In [7]:
# Sanity check case study pairs
has_overlap = df["dyad_id"].isin(dyads_case_study).any()

if has_overlap:
  print("⚠️ Some case study dyads are present in the DataFrame.")
else:
  print("✅ No case study dyads found in the DataFrame.")

✅ No case study dyads found in the DataFrame.


In [8]:
if SUBSAMPLE_ENABLED:
  dyad_subsample = pd.Series(df["dyad_id"].unique()).sample(n=N_DYADS, random_state=RANDOM_SEED, replace=False)
  df = df[df["dyad_id"].isin(dyad_subsample)]
print(f"Unique dyads: {df["dyad_id"].nunique()}")

Unique dyads: 10


In [9]:
df["sanction"] = (df[SANCTION_COLS]
                  .sum(axis=1)).astype(int)

In [10]:
num_cols = ["distw", "GDP_reporter", "GDP_partner", "sanction", "contig",
            "comlang_off", "colony", "smctry", "Year", ]
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce").astype(float)
df = df.dropna(subset=num_cols)

In [11]:
df["Year"] = df["Year"].astype(int)
for col in ["dyad_id"]:
  df[col] = pd.Categorical(df[col], categories=sorted(df[col].unique()))

In [12]:
# Save EXPORT std and median to undo centering
EXPORT_STD = df["EXPORT"].std()
EXPORT_MEDIAN = df["EXPORT"].median()

In [13]:
center_columns = ["distw", "GDP_reporter", "GDP_partner", "EXPORT"]
for col in center_columns:
  median = df[col].median()
  std_df = df[col].std()
  df[col + "_centered"] = (df[col] - median) / std_df
FEATURES += ["distw_centered"]

In [14]:
lag_cols = ["GDP_reporter_centered", "GDP_partner_centered", "sanction"]
for col in lag_cols:
  for index in range(1, N_LAGS + 1):
    df[f"{col}_lag{index}"] = df.groupby("dyad_id", observed=True)[col].shift(index)

In [15]:
df = df.dropna()
FEATURES += [f"{c}_lag{index}" for c in lag_cols for index in range(1, N_LAGS + 1)]

## Split data

In [16]:
# Embeddings
dyad_to_idx = { dyad: i for i, dyad in enumerate(df["dyad_id"].cat.categories) }
df["dyad_idx"] = df["dyad_id"].map(dyad_to_idx).astype(int)

In [17]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)

train_idx, test_idx = next(gss.split(df, groups=df["dyad_id"]))
test_df = df.iloc[test_idx]
train_df = df.iloc[train_idx]

train_idx, val_idx = next(gss.split(train_df, groups=train_df["dyad_id"]))
val_df = train_df.iloc[val_idx]
train_df = train_df.iloc[train_idx]

In [18]:
train_df.loc[:, FEATURES] = train_df.loc[:, FEATURES].astype(
  "float32",
  copy=False
)

# Train

## Define Fold and Epoch steps
_For reusability_

In [19]:
# Create KFold object
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)

In [ ]:
def epoch_step(
    model: nn.Module,
    optimizer: Optimizer,
    criterion: nn.Module,
    scheduler: LRScheduler,
    train_loader: DataLoader,
    val_loader: DataLoader,
    device: any,
    scaler: torch.amp.GradScaler,
    epoch: int,
) -> float:
  # =========================
  # TRAIN
  # =========================
  model.train()
  train_loss_sum = 0.0
  train_count = 0

  for X, y, di in train_loader:
    X, y, di = map(lambda t: t.to(device, non_blocking=True), (X, y, di))

    optimizer.zero_grad(set_to_none=True)

    with autocast("cuda", dtype=torch.bfloat16):
      y_pred = model(X, di)
      if not torch.isfinite(y_pred).all():
        print("NaN/Inf in y_pred — stopping fold")
        return float("inf")

      loss = criterion(y_pred, y)

    if not torch.isfinite(loss):
      print("NaN/Inf loss — stopping fold")
      return float("inf")

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    clip_grad_norm_(model.parameters(), 0.3)

    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

    train_loss_sum += loss.item()
    train_count += 1

  train_loss = train_loss_sum / train_count

  # =========================
  # VALIDATION
  # =========================
  model.eval()
  val_loss_sum = 0.0
  val_count = 0

  with torch.no_grad():
    for X, y, di in val_loader:
      X, y, di = map(lambda t: t.to(device, non_blocking=True), (X, y, di))
      with autocast("cuda", dtype=torch.bfloat16):
        preds = model(X, di)
        val_loss_sum += criterion(preds, y).item()
        val_count += 1

  val_rmse = math.sqrt(val_loss_sum / val_count)

  with open(LOSS_LOG_PATH, "a") as f:
    f.write(f"{epoch},{train_loss},{val_rmse}\n")

  return val_rmse

In [ ]:
def _make_loader(subset, *, batch_size: int, shuffle: bool, n_workers: int) -> DataLoader:
  """Colab-friendly DataLoader with aggressive prefetch + pinned memory."""
  kwargs = dict(
    batch_size=batch_size,
    shuffle=shuffle,
    num_workers=n_workers,
    pin_memory=True,
    persistent_workers=(n_workers > 0),
  )

  # Only valid when num_workers > 0
  if n_workers > 0:
    kwargs["prefetch_factor"] = 4

  return DataLoader(subset, **kwargs)

In [ ]:
# Define fold step
def fold_step(
    fold: int,
    train_idx: List,
    val_idx: List,
    dataset: Dataset,
    batch_size: int,
    num_epochs: int,
    model: nn.Module,
    device: any,
    optimizer: Optimizer,
    criterion: nn.Module,
    serial: str,
):
  # Use all available cores
  n_workers = min(8, (os.cpu_count() or 2))

  train_loader = _make_loader(Subset(dataset, train_idx), batch_size=batch_size, shuffle=True, n_workers=n_workers)
  val_loader = _make_loader(Subset(dataset, val_idx), batch_size=batch_size, shuffle=False, n_workers=n_workers)

  scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=7e-4,
    epochs=NUM_EPOCHS,
    steps_per_epoch=len(train_loader),
    div_factor=25,
    final_div_factor=100,
    pct_start=0.3,
    anneal_strategy="cos",
    three_phase=False,
  )

  ckpt_file = ckpt_path(serial, fold)
  tmp_ckpt_file = ckpt_file + ".tmp"

  ckpt = safe_load_ckpt(ckpt_file, map_location=device)

  start_epoch = 0
  best_rmse = float("inf")
  best_state = copy.deepcopy(model.state_dict())
  scaler = GradScaler("cuda", enabled=USE_GRAD_SCALER)

  if ckpt is not None:
    print(f"🔄 Resuming from checkpoint: {ckpt_file}")

    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    scaler.load_state_dict(ckpt["scaler_state"])

    start_epoch = ckpt["epoch"] + 1
    best_rmse = ckpt["best_rmse"]
    best_state = ckpt["best_state"]

  print(f"Start epoch train for fold {fold}")

  for epoch in range(start_epoch, num_epochs):
    val_rmse = epoch_step(
      model=model,
      optimizer=optimizer,
      criterion=criterion,
      scheduler=scheduler,
      train_loader=train_loader,
      val_loader=val_loader,
      device=device,
      scaler=scaler,
      epoch=epoch
    )

    print(f"Epoch {epoch + 1:02d}/{num_epochs} | val RMSE: {val_rmse:.4f}")

    if val_rmse < best_rmse - 1e-4:
      best_rmse = val_rmse
      best_state = copy.deepcopy(model.state_dict())

    # =========================
    # ATOMIC CHECKPOINT SAVE
    # =========================
    state = {
      "epoch": epoch,
      "model_state": model.state_dict(),
      "optimizer_state": optimizer.state_dict(),
      "scheduler_state": scheduler.state_dict(),
      "scaler_state": scaler.state_dict(),
      "best_rmse": best_rmse,
      "best_state": best_state,
    }

    torch.save(state, tmp_ckpt_file)
    os.replace(tmp_ckpt_file, ckpt_file)

  # =========================
  # FINAL EVALUATION
  # =========================
  model.load_state_dict(best_state)
  model.eval()

  preds_centered, truth_centered = [], []

  with torch.inference_mode():
    for X, y, di in val_loader:
      X, di = map(lambda t: t.to(device, non_blocking=True), (X, di))

      with autocast("cuda", enabled=(getattr(device, "type", "cpu") == "cuda" and USE_AMP)):
        out = model(X, di)

        preds_centered.append(out.float().cpu())
        truth_centered.append(y)

  preds_centered = torch.cat(preds_centered).numpy()
  truth_centered = torch.cat(truth_centered).numpy()

  # Convert back to raw scale
  preds_raw = preds_centered * EXPORT_STD + EXPORT_MEDIAN
  truth_raw = truth_centered * EXPORT_STD + EXPORT_MEDIAN

  rmse_centered = tu.rmse(truth_centered, preds_centered)
  mae_centered = tu.mae(truth_centered, preds_centered)
  rmae_centered = tu.rmae(truth_centered, preds_centered)
  pseudo_r2_centered = tu.pseudo_r2(truth_centered, preds_centered)

  print(
    f"Fold {fold} CENTERED  RMSE {rmse_centered:.4f} | "
    f"MAE {mae_centered:.4f} | R² {pseudo_r2_centered:.4f} | "
    f"RMAE {rmae_centered:.4f}"
  )

  rmse_raw = tu.rmse(truth_raw, preds_raw)
  mae_raw = tu.mae(truth_raw, preds_raw)
  rmae_raw = tu.rmae(truth_raw, preds_raw)
  pseudo_r2_raw = tu.pseudo_r2(truth_raw, preds_raw)

  print(
    f"Fold {fold} RAW  RMSE {rmse_raw:.4f} | "
    f"MAE {mae_raw:.4f} | R² {pseudo_r2_raw:.4f} | "
    f"RMAE {rmae_raw:.4f}"
  )

  return (
    { "RMSE": rmse_centered, "MAE": mae_centered, "R2": pseudo_r2_centered, "RMAE": rmae_centered },
    { "RMSE": rmse_raw, "MAE": mae_raw, "R2": pseudo_r2_raw, "RMAE": rmae_raw },
    copy.deepcopy(best_state),
  )

## Raw dataset

### Split dataset

In [22]:
dataset, dyad_to_idx = tu.make_panel_datasets_dyad(
  data=df,
  features=FEATURES,
  target=TARGET,
  horizon=HORIZON,
)

In [23]:
# Create DataLoaders for the 3 sets
train_loader = DataLoader(
  Subset(dataset, train_idx),
  batch_size=BATCH_SIZE,
  shuffle=True,
  num_workers=10,
  persistent_workers=True,
  prefetch_factor=2,
  pin_memory=False
)

val_loader = DataLoader(
  Subset(dataset, val_idx),
  batch_size=BATCH_SIZE,
  shuffle=False,
  num_workers=10,
  persistent_workers=True,
  prefetch_factor=2,
  pin_memory=False
)

test_loader = DataLoader(
  Subset(dataset, test_idx),
  batch_size=BATCH_SIZE,
  shuffle=False,
  num_workers=10,
  persistent_workers=True,
  prefetch_factor=2,
  pin_memory=False
)

In [ ]:
# Create DataLoaders for the 3 sets
n_workers = min(8, (os.cpu_count() or 2))

train_kwargs = dict(
  batch_size=BATCH_SIZE,
  shuffle=True,
  num_workers=n_workers,
  persistent_workers=(n_workers > 0),
  pin_memory=True,
)
val_kwargs = dict(
  batch_size=BATCH_SIZE,
  shuffle=False,
  num_workers=n_workers,
  persistent_workers=(n_workers > 0),
  pin_memory=True,
)
test_kwargs = dict(
  batch_size=BATCH_SIZE,
  shuffle=False,
  num_workers=n_workers,
  persistent_workers=(n_workers > 0),
  pin_memory=True,
)

if n_workers > 0:
  train_kwargs["prefetch_factor"] = 4
  val_kwargs["prefetch_factor"] = 4
  test_kwargs["prefetch_factor"] = 4

train_loader = DataLoader(Subset(dataset, train_idx), **train_kwargs)
val_loader = DataLoader(Subset(dataset, val_idx), **val_kwargs)
test_loader = DataLoader(Subset(dataset, test_idx), **test_kwargs)

### Train model

In [24]:
# Save best train iteration
best_fold_state = None
best_fold_rmse = float("inf")
metrics_per_fold = {
  "RMSE": [],
  "MAE": [],
  "R2": [],
  "RMAE": [],
}

metrics_per_fold_raw = {
  "RMSE": [],
  "MAE": [],
  "R2": [],
  "RMAE": [],
}

In [ ]:
for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(len(dataset))), 1):

  ckpt_file = ckpt_path(SERIAL_NUMBER, fold)
  ckpt = safe_load_ckpt(ckpt_file, map_location="cpu")

  if ckpt is not None and ckpt.get("epoch", -1) >= NUM_EPOCHS - 1:
    print(f"✅ Fold {fold} already completed — skipping.")
    continue

  print(f"=== FOLD {fold}/{N_SPLITS} ===")

  model = tu.DyadGRU(
    n_features=len(FEATURES),
    n_layers=N_LAYERS,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
    horizon=HORIZON,
    n_dyads=len(dyad_to_idx),
    embed_dim=EMBEDDING_SIZE
  ).to(device=device)

  # JIT Compile the model for speedup (requires PyTorch 2.0+)
  if USE_TORCH_COMPILE and hasattr(torch, "compile") and device.type == "cuda":
    try:
      model = torch.compile(model, mode=COMPILE_MODE)
      print(f"Model compiled with torch.compile(mode={COMPILE_MODE!r})")
    except Exception as e:
      print(f"Could not compile model: {e}")

  criterion = nn.SmoothL1Loss(beta=0.5)

  adamw_kwargs = dict(lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=(0.9, 0.95), eps=1e-8)
  if device.type == "cuda":
    try:
      optimizer = optim.AdamW(model.parameters(), **adamw_kwargs, fused=True)
      print("Using fused AdamW")
    except TypeError:
      optimizer = optim.AdamW(model.parameters(), **adamw_kwargs)
  else:
    optimizer = optim.AdamW(model.parameters(), **adamw_kwargs)

  fold_metrics, fold_metrics_raw, best_state = fold_step(fold=fold,
                                                         train_idx=train_idx,
                                                         val_idx=val_idx,
                                                         dataset=dataset,
                                                         batch_size=BATCH_SIZE,
                                                         num_epochs=NUM_EPOCHS,
                                                         model=model,
                                                         device=device,
                                                         optimizer=optimizer,
                                                         criterion=criterion,
                                                         serial=SERIAL_NUMBER)

  if fold_metrics["RMSE"] < best_fold_rmse:
    best_fold_rmse = fold_metrics["RMSE"]
    best_fold_state = copy.deepcopy(best_state)

  for k, v in fold_metrics.items():
    metrics_per_fold[k].append(v)

  for k, v in fold_metrics_raw.items():
    metrics_per_fold_raw[k].append(v)

## Save Model

In [26]:
if SAVE_ENABLED:
  torch.save({
    "model_state_dict": best_fold_state,
    "model_hyperparams": {
      "n_features": len(FEATURES),
      "n_dyads": len(dyad_to_idx),
      "embed_dim": EMBEDDING_SIZE,
      "hidden_size": HIDDEN_SIZE,
      "n_layers": N_LAYERS,
      "dropout": DROPOUT,
      "horizon": HORIZON,
    },
    "dyad_to_idx": dyad_to_idx,
    "feature_names": FEATURES,
    "scaler": { "EXPORT_MEDIAN": EXPORT_MEDIAN, "EXPORT_STD": EXPORT_STD },
  }, PATH_TO_FOLDER + SERIAL_NUMBER + ".pt")
print("Saved model to ", PATH_TO_FOLDER + SERIAL_NUMBER + ".pt")


Saved model to  ../../models/GRU-0_0003lr-0_05d-256hs-0wd-128bs-2layers-128es-kfolds8-hp.pt


In [27]:
def summarize(xs):
  xs = np.asarray(xs, dtype=float)
  n = xs.size
  mean = xs.mean()
  std = xs.std(ddof=1)  # sample std
  se = std / math.sqrt(n)
  try:
    from scipy.stats import t
    tcrit = t.ppf(0.975, df=n - 1)
  except Exception:
    tcrit = 1.96  # normal approx≈
  ci95 = tcrit * se
  return mean, std, ci95

In [28]:
print("\n=== Cross-fold CENTERED summary ===")
for name in ["MAE", "RMSE", "R2", "RMAE"]:
  mean, std, ci = summarize(metrics_per_fold[name])
  print(f"{name:>5}: {mean:.4f} ± {std:.4f}  (95% CI ±{ci:.4f})")

print("\n=== Cross-fold RAW summary ===")
for name in ["MAE", "RMSE", "R2", "RMAE"]:
  mean, std, ci = summarize(metrics_per_fold_raw[name])
  print(f"{name:>5}: {mean:.4f} ± {std:.4f}  (95% CI ±{ci:.4f})")


=== Cross-fold CENTERED summary ===
  MAE: 0.2157 ± 0.1515  (95% CI ±0.1267)
 RMSE: 0.7063 ± 0.4701  (95% CI ±0.3930)
   R2: 0.5120 ± 0.1939  (95% CI ±0.1621)
 RMAE: 0.7717 ± 0.1635  (95% CI ±0.1367)

=== Cross-fold RAW summary ===
  MAE: 27056.0008 ± 19011.8927  (95% CI ±15894.3400)
 RMSE: 88605.0465 ± 58976.5991  (95% CI ±49305.6708)
   R2: 0.5120 ± 0.1939  (95% CI ±0.1621)
 RMAE: 0.7717 ± 0.1635  (95% CI ±0.1367)
